# Fase 17D — Smoke tests CKKS dos três datasets

Executa uma rodada federada com atualizações cifradas por CKKS, agregação no domínio cifrado e decifragem pelo Servidor A. Usa os mesmos inputs congelados do Baseline e compara a atualização CKKS com uma sombra plaintext. Resultados de smoke não são utilizáveis na dissertação.

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable,'-m','pip','install','-q','tenseal'])
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, time
import numpy as np
import pandas as pd
import psutil, torch, tenseal as ts
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
CAMPAIGN_ID='THESIS_OFFICIAL_CAMPAIGN_V2_20260901'; PROJECT=Path('/content/drive/MyDrive/Mestrado_Criptografia'); CAMPAIGN=PROJECT/'OFFICIAL_CAMPAIGN_V2'/CAMPAIGN_ID
FREEZE=CAMPAIGN/'01_DATASET_FREEZE'; CONTROL=CAMPAIGN/'00_CAMPAIGN_CONTROL'; SMOKE=CAMPAIGN/'03_SMOKE_TESTS'
SEED=42; CLIENTS=5; LOCAL_EPOCHS=3; BATCH_SIZE=64; LEARNING_RATE=0.1; THRESHOLD=0.5
CKKS={'poly_modulus_degree':8192,'coeff_mod_bit_sizes':[60,40,40,60],'global_scale':2**40,'slots':4096,'library':'TenSEAL/Microsoft SEAL'}
torch.set_num_threads(min(2,os.cpu_count() or 1)); np.random.seed(SEED); torch.manual_seed(SEED)
phase17c=json.loads((CONTROL/'PHASE17C_MASTER_GATE.json').read_text()); assert phase17c['all_baseline_smokes_approved'] is True
print('TenSEAL:',getattr(ts,'__version__','unknown')); print('CKKS smoke tests autorizados.')

In [ ]:
def sha256_file(p):
 h=hashlib.sha256()
 with Path(p).open('rb') as f:
  for b in iter(lambda:f.read(8*1024*1024),b''): h.update(b)
 return h.hexdigest()
def load_split(path):
 z=np.load(path,allow_pickle=True); out={}
 for k in z.files:
  lk=k.lower()
  if 'train' in lk and 'client' not in lk: out['train']=np.asarray(z[k]).reshape(-1).astype(int)
  elif ('val' in lk or 'valid' in lk) and 'client' not in lk: out['validation']=np.asarray(z[k]).reshape(-1).astype(int)
  elif 'test' in lk and 'client' not in lk: out['test']=np.asarray(z[k]).reshape(-1).astype(int)
 return out
def load_clients(path):
 z=np.load(path,allow_pickle=True); return [np.asarray(z[f'client_{i}']).reshape(-1).astype(int) for i in range(CLIENTS)]
def apply_frozen_preprocessing(X,path):
 p=np.load(path,allow_pickle=True); median=p['median']; mean=p['mean']; std=p['std']; X=np.asarray(X,dtype=np.float64); filled=np.where(np.isfinite(X),X,median); Z=((filled-mean)/std).astype(np.float32); assert np.isfinite(Z).all(); return Z
def metrics(y,prob):
 pred=(prob>=THRESHOLD).astype(int); tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); return {'accuracy':float(accuracy_score(y,pred)),'precision':float(precision_score(y,pred,zero_division=0)),'recall':float(recall_score(y,pred,zero_division=0)),'f1':float(f1_score(y,pred,zero_division=0)),'auroc':float(roc_auc_score(y,prob)) if len(np.unique(y))==2 else None,'tn':int(tn),'fp':int(fp),'fn':int(fn),'tp':int(tp)}
def predict(X,state):
 w=state[:-1]; b=state[-1]; logits=X@w+b; return 1/(1+np.exp(-np.clip(logits,-40,40)))
def train_client_delta(X,y,idx,initial,client_id):
 w0=initial[:-1].astype(np.float32); b0=initial[-1:].astype(np.float32); model=torch.nn.Linear(X.shape[1],1); model.weight.data.copy_(torch.from_numpy(w0.reshape(1,-1))); model.bias.data.copy_(torch.from_numpy(b0)); opt=torch.optim.SGD(model.parameters(),lr=LEARNING_RATE); loss_fn=torch.nn.BCEWithLogitsLoss(); rng=np.random.default_rng(SEED+client_id); start=time.perf_counter()
 for epoch in range(LOCAL_EPOCHS):
  order=np.asarray(idx).copy(); rng.shuffle(order)
  for s in range(0,len(order),BATCH_SIZE):
   batch=order[s:s+BATCH_SIZE]; xb=torch.from_numpy(X[batch]); yb=torch.from_numpy(y[batch].astype(np.float32)).reshape(-1,1); opt.zero_grad(); loss=loss_fn(model(xb),yb); loss.backward(); opt.step()
 state=np.concatenate([model.weight.detach().numpy().reshape(-1),model.bias.detach().numpy().reshape(-1)]).astype(np.float64); return state-initial.astype(np.float64),time.perf_counter()-start


In [ ]:
DAHL_FEATURES=['mean','std','min','max','median','q05','q25','q75','q95','rms','range','mean_abs_diff']
def prepare(dataset):
 root=FREEZE/dataset/'SCIENTIFIC_FREEZE'
 if dataset=='PHYSIONET_CHALLENGE_2012':
  df=pd.read_csv(root/'physionet_challenge_2012_features.csv'); cols=[c for c in df.columns if c not in {'RecordID','target'}]; X=df[cols].apply(pd.to_numeric,errors='coerce').to_numpy(); y=pd.to_numeric(df.target).to_numpy(dtype=int); split=load_split(root/'official_split_seed42.npz')
 elif dataset=='DAHL_RATS':
  df=pd.read_csv(root/'dahl_derived_features.csv'); X=df[DAHL_FEATURES].apply(pd.to_numeric,errors='coerce').to_numpy(); y=pd.to_numeric(df.target).to_numpy(dtype=int); split=load_split(root/'official_split_seed42.npz')
 else:
  z=np.load(root/'chexchonet_embeddings.npz',allow_pickle=True); X=np.asarray(z['embeddings']); y=np.asarray(z['targets']).astype(int); raw=np.asarray(np.load(root/'official_split_original.npy',allow_pickle=True)).reshape(-1); norm=lambda v:{'0':'train','1':'validation','2':'test','val':'validation','valid':'validation','dev':'validation'}.get(str(v).strip().lower(),str(v).strip().lower()); labels=np.array([norm(v) for v in raw]); split={s:np.where(labels==s)[0] for s in ['train','validation','test']}
 X=apply_frozen_preprocessing(X,root/'preprocessing_train_only.npz'); clients=load_clients(root/'official_client_partition_noniid_seed42.npz'); initial=np.load(root/'shared_initial_state.npy').reshape(-1).astype(np.float64); assert len(initial)==X.shape[1]+1; return root,X,y,split,clients,initial


In [ ]:
def create_contexts():
 secret=ts.context(ts.SCHEME_TYPE.CKKS,poly_modulus_degree=CKKS['poly_modulus_degree'],coeff_mod_bit_sizes=CKKS['coeff_mod_bit_sizes']); secret.global_scale=CKKS['global_scale']; secret.generate_galois_keys(); public_bytes=secret.serialize(save_public_key=True,save_secret_key=False,save_galois_keys=True,save_relin_keys=True); public=ts.context_from(public_bytes); return secret,public,len(public_bytes)
def run_ckks_smoke(dataset):
 root,X,y,split,clients,initial=prepare(dataset); timestamp=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'); run_id=f'RUN__{dataset}__CKKS__SMOKE__R1__C5__S42__{timestamp}'; run_dir=SMOKE/dataset/'CKKS'/run_id; run_dir.mkdir(parents=True,exist_ok=False)
 status={'run_id':run_id,'campaign_id':CAMPAIGN_ID,'dataset':dataset,'scenario':'CKKS','execution_type':'SMOKE','status':'RUNNING','usable_in_thesis':False,'started_at_utc':datetime.now(timezone.utc).isoformat()}; (run_dir/'RUN_STATUS.json').write_text(json.dumps(status,indent=2))
 secret,public,public_context_bytes=create_contexts(); deltas=[]; encrypted=[]; client_rows=[]; round_start=time.perf_counter(); mem0=psutil.Process().memory_info().rss
 for i,idx in enumerate(clients):
  delta,train_s=train_client_delta(X,y,idx,initial,i); enc0=time.perf_counter(); ct=ts.ckks_vector(public,delta.tolist()); enc_s=time.perf_counter()-enc0; blob=ct.serialize(); deltas.append((delta,len(idx))); encrypted.append((ct,len(idx),len(blob))); client_rows.append({'client_id':i,'samples':len(idx),'train_seconds':train_s,'encryption_seconds':enc_s,'ciphertext_bytes':len(blob)}); print(dataset,'cliente',i,'cifrado:',len(blob),'bytes')
 total=sum(n for _,n in deltas); plain_delta=sum(d*n for d,n in deltas)/total
 agg0=time.perf_counter(); aggregate=None
 for ct,n,_ in encrypted:
  weighted=ct*(n/total); aggregate=weighted if aggregate is None else aggregate+weighted
 aggregation_s=time.perf_counter()-agg0; aggregate_blob=aggregate.serialize(); dec0=time.perf_counter(); aggregate_secret=ts.ckks_vector_from(secret,aggregate_blob); decrypted=np.asarray(aggregate_secret.decrypt(),dtype=np.float64)[:len(initial)]; decryption_s=time.perf_counter()-dec0
 ckks_state=initial+decrypted; shadow_state=initial+plain_delta; error=decrypted-plain_delta; val_prob=predict(X[split['validation']],ckks_state); shadow_prob=predict(X[split['validation']],shadow_state); val_metrics=metrics(y[split['validation']],val_prob); disagreement=float(np.mean((val_prob>=THRESHOLD)!=(shadow_prob>=THRESHOLD)))
 parameter_bytes=int(len(initial)*4); communication={'model_download_bytes':CLIENTS*parameter_bytes,'client_ciphertexts_bytes':sum(x[2] for x in encrypted),'aggregated_ciphertext_bytes':len(aggregate_blob),'decrypted_update_bytes':parameter_bytes}; communication['total_bytes']=sum(communication.values())
 numeric={'max_abs_error':float(np.max(np.abs(error))),'mean_abs_error':float(np.mean(np.abs(error))),'rmse':float(np.sqrt(np.mean(error**2))),'validation_probability_max_abs_difference':float(np.max(np.abs(val_prob-shadow_prob))),'validation_prediction_disagreement_rate':disagreement}
 checks={'five_clients_completed':len(client_rows)==5,'parameter_count_preserved':len(decrypted)==len(initial),'finite_decryption':bool(np.isfinite(decrypted).all()),'max_abs_error_le_1e_3':numeric['max_abs_error']<=1e-3,'mean_abs_error_le_1e_4':numeric['mean_abs_error']<=1e-4,'prediction_disagreement_le_1e_3':disagreement<=1e-3,'test_not_used':True}; approved=all(checks.values())
 round_metrics={'round':1,'wall_seconds':time.perf_counter()-round_start,'rss_start_bytes':int(mem0),'rss_end_bytes':int(psutil.Process().memory_info().rss),'encryption_total_seconds':sum(r['encryption_seconds'] for r in client_rows),'aggregation_seconds':aggregation_s,'decryption_seconds':decryption_s,'numeric_error':numeric,'communication':communication,'validation':val_metrics,'clients':client_rows}; (run_dir/'round_001_metrics.json').write_text(json.dumps(round_metrics,indent=2)); pd.DataFrame(client_rows).to_csv(run_dir/'client_metrics.csv',index=False); np.savez_compressed(run_dir/'final_state.npz',state=ckks_state,shadow_state=shadow_state)
 config={'dataset':dataset,'scenario':'CKKS','execution_type':'SMOKE','seed':SEED,'rounds':1,'clients':CLIENTS,'local_epochs':LOCAL_EPOCHS,'batch_size':BATCH_SIZE,'learning_rate':LEARNING_RATE,'model':'LogisticRegression','parameter_count':len(initial),'aggregation':'CKKS_FedAvg_weighted_by_client_samples','ckks':CKKS,'public_context_bytes':public_context_bytes,'test_evaluated':False,'client_partition_sha256':sha256_file(root/'official_client_partition_noniid_seed42.npz'),'initial_state_sha256':sha256_file(root/'shared_initial_state.npy'),'preprocessing_sha256':sha256_file(root/'preprocessing_train_only.npz')}; (run_dir/'RUN_CONFIG.json').write_text(json.dumps(config,indent=2))
 status.update({'status':'COMPLETED_APPROVED' if approved else 'COMPLETED_REJECTED','completed_at_utc':datetime.now(timezone.utc).isoformat(),'smoke_approved':approved,'checks':checks}); (run_dir/'RUN_STATUS.json').write_text(json.dumps(status,indent=2)); return {'dataset':dataset,'run_id':run_id,'run_dir':str(run_dir),'approved':approved,'parameter_count':len(initial),'validation':val_metrics,'numeric_error':numeric,'communication':communication,'wall_seconds':round_metrics['wall_seconds'],'checks':checks}
results=[run_ckks_smoke(d) for d in ['PHYSIONET_CHALLENGE_2012','DAHL_RATS','CHEXCHONET']]
print(json.dumps(results,indent=2,ensure_ascii=False))

In [ ]:
all_ok=all(r['approved'] for r in results); master={'phase':'17D','campaign_id':CAMPAIGN_ID,'completed_at_utc':datetime.now(timezone.utc).isoformat(),'ckks_smoke_approvals':{r['dataset']:r['approved'] for r in results},'all_ckks_smokes_approved':all_ok,'official_training_authorized':False,'next_authorized_step':'FASE_17E_HYBRID_SMOKE_TESTS' if all_ok else 'REPAIR_CKKS_SMOKE_FAILURES','results':results}; (CONTROL/'PHASE17D_MASTER_GATE.json').write_text(json.dumps(master,indent=2,ensure_ascii=False)); status=json.loads((CONTROL/'CAMPAIGN_STATUS.json').read_text()); status.update({'status':'PHASE17D_COMPLETED' if all_ok else 'PHASE17D_BLOCKED','phase17d_ckks_smokes_approved':all_ok,'next_authorized_step':master['next_authorized_step']}); (CONTROL/'CAMPAIGN_STATUS.json').write_text(json.dumps(status,indent=2,ensure_ascii=False)); print('='*100); print(json.dumps(master,indent=2,ensure_ascii=False)); print('='*100)

In [ ]:
export=Path('/content/PHASE17D_EVIDENCE'); shutil.rmtree(export,ignore_errors=True); export.mkdir(); [shutil.copy2(CONTROL/n,export/n) for n in ['CAMPAIGN_STATUS.json','PHASE17C_MASTER_GATE.json','PHASE17D_MASTER_GATE.json']]
for r in results:
 dst=export/r['dataset']; shutil.copytree(Path(r['run_dir']),dst,ignore=shutil.ignore_patterns('final_state.npz'))
zip_path=shutil.make_archive('/content/PHASE17D_EVIDENCE','zip','/content','PHASE17D_EVIDENCE'); permanent=CONTROL/'EXPORTS'/'PHASE17D_EVIDENCE.zip'; permanent.parent.mkdir(parents=True,exist_ok=True); shutil.copy2(zip_path,permanent); print('Salvo em:',permanent); files.download(str(permanent))